# Recommender Modelling and Offline Evaluation

This notebook builds and evaluates recommendation strategies using the reproducible chronological datasets created in `03_data_preparation.ipynb`.

The modelling workflow is:

1. load and verify the processed modelling tables;
2. establish a non-personalized popularity baseline;
3. build a personalized collaborative recommendation model;
4. add product-content information for a hybrid recommender;
5. tune model settings using validation targets only;
6. compare ranking quality, catalogue coverage, popularity bias, and recommendation diversity;
7. select one configuration;
8. retrain it using the final-training data;
9. evaluate it once against the untouched test targets;
10. save the selected recommender artifacts for the Streamlit application.

The intended final system is an explainable hybrid video-game recommender with:

- collaborative signals from user behaviour;
- content signals from product metadata;
- a popularity fallback for users or products with limited history;
- human-readable explanations for individual recommendations.

The validation targets are used for model comparison and configuration selection. The test targets remain untouched until the final model and its settings have been selected.

In [23]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)
from sklearn.neighbors import NearestNeighbors

import time

import joblib

## Load and verify the modelling data

The modelling notebook reads the processed tables created by the data-preparation notebook.

Loading saved tables instead of rebuilding the split ensures that every recommender uses the same verified training, validation, and test data. The row-count and uniqueness assertions below protect against loading the wrong files or accidentally changing the evaluation protocol.

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_data_directory = (
    project_root
    / "data"
    / "processed"
)

development_training_path = (
    processed_data_directory
    / "development_positive_training.csv.gz"
)

final_training_path = (
    processed_data_directory
    / "final_positive_training.csv.gz"
)

validation_targets_path = (
    processed_data_directory
    / "validation_targets.csv.gz"
)

test_targets_path = (
    processed_data_directory
    / "test_targets.csv.gz"
)

development_training = pd.read_csv(
    development_training_path,
    parse_dates=["interaction_time"],
)

final_training = pd.read_csv(
    final_training_path,
    parse_dates=["interaction_time"],
)

validation_targets = pd.read_csv(
    validation_targets_path,
    parse_dates=["interaction_time"],
)

test_targets = pd.read_csv(
    test_targets_path,
    parse_dates=["interaction_time"],
)

assert len(development_training) == 478_619
assert len(final_training) == 566_743
assert len(validation_targets) == 88_124
assert len(test_targets) == 88_124

assert validation_targets["user_id"].is_unique
assert test_targets["user_id"].is_unique

assert not (
    set(validation_targets["interaction_id"])
    & set(test_targets["interaction_id"])
)

print("Project root:", project_root)
print("Processed-data directory:", processed_data_directory)
print()
print(
    "Development-training rows:",
    f"{len(development_training):,}",
)
print(
    "Final-training rows:",
    f"{len(final_training):,}",
)
print(
    "Validation-target rows:",
    f"{len(validation_targets):,}",
)
print(
    "Test-target rows:",
    f"{len(test_targets):,}",
)
print(
    "Unique development users:",
    f"{development_training['user_id'].nunique():,}",
)
print(
    "Unique development products:",
    f"{development_training['parent_asin'].nunique():,}",
)

display(
    development_training.head()
)

Project root: /Users/trevaogwang/Documents/mobile-projects/saidia_recommeder
Processed-data directory: /Users/trevaogwang/Documents/mobile-projects/saidia_recommeder/data/processed

Development-training rows: 478,619
Final-training rows: 566,743
Validation-target rows: 88,124
Test-target rows: 88,124
Unique development users: 93,949
Unique development products: 25,319


,user_id,parent_asin,rating,timestamp,interaction_time,interaction_id,is_positive
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,B082R1RGZF,5.0,1563320217391,2019-07-16 23:36:57.391000+00:00,0,True
1,AE222HFZDH6BPTYFOUWGGU63YSIQ,B07SNN8GV5,5.0,1576420340876,2019-12-15 14:32:20.876000+00:00,1,True
2,AE222HFZDH6BPTYFOUWGGU63YSIQ,B01GY35T4S,5.0,1578161337203,2020-01-04 18:08:57.203000+00:00,2,True
3,AE2252DKW4XJIZP5QPFMQVJBVRTA,B0050SX4CI,4.0,1352359389000,2012-11-08 07:23:09+00:00,5,True
4,AE2252DKW4XJIZP5QPFMQVJBVRTA,B002ORTCAQ,5.0,1357679634000,2013-01-08 21:13:54+00:00,6,True


## Popularity baseline

The first recommender is a non-personalized popularity baseline.

Products are ranked by their number of positive interactions in the development-training data. For each user, products already present in that user's development history are excluded before the top recommendations are returned.

This baseline provides an essential benchmark:

- it is simple and inexpensive;
- it often performs surprisingly well on popularity-skewed datasets;
- a more complex personalized model should demonstrate measurable improvement over it;
- it provides a fallback strategy for users with limited behavioural history.

Because the model learns popularity only from development training, the validation and test targets remain hidden.

In [3]:
product_popularity = (
    development_training
    .groupby("parent_asin")
    .size()
    .rename("positive_interaction_count")
    .sort_values(ascending=False)
    .reset_index()
)

popularity_product_order = (
    product_popularity["parent_asin"].tolist()
)

development_seen_by_user = (
    development_training
    .groupby("user_id")["parent_asin"]
    .agg(set)
    .to_dict()
)

assert product_popularity["parent_asin"].is_unique

print(
    "Products in popularity ranking:",
    f"{len(product_popularity):,}",
)

print(
    "Users with stored development history:",
    f"{len(development_seen_by_user):,}",
)

display(
    product_popularity.head(10)
)

Products in popularity ranking: 25,319
Users with stored development history: 93,949


,parent_asin,positive_interaction_count
0,B01N3ASPNV,1752
1,B07YBXFDYN,1510
2,B0086VPUHI,1480
3,B00BGA9WK2,1267
4,B000N5Z2L4,1233
5,B004RMK5QG,1128
6,B00BN5T30E,1098
7,B077GG9D5D,972
8,B07YBWT3PK,965
9,B087NNPYP3,938


In [4]:
recommendation_count = 10


def recommend_popular_products(
    user_id,
    number_of_recommendations=10,
):
    seen_products = development_seen_by_user.get(
        user_id,
        set(),
    )

    recommendations = []

    for product_id in popularity_product_order:
        if product_id not in seen_products:
            recommendations.append(product_id)

        if len(recommendations) == number_of_recommendations:
            break

    return recommendations


validation_evaluation = (
    validation_targets[
        [
            "user_id",
            "parent_asin",
        ]
    ]
    .rename(
        columns={
            "parent_asin": "target_product_id"
        }
    )
    .copy()
)

validation_evaluation["recommendations"] = [
    recommend_popular_products(
        user_id,
        recommendation_count,
    )
    for user_id in validation_evaluation["user_id"]
]


def find_target_rank(
    target_product_id,
    recommendations,
):
    if target_product_id not in recommendations:
        return np.nan

    return (
        recommendations.index(target_product_id)
        + 1
    )


validation_evaluation["target_rank"] = [
    find_target_rank(
        target_product_id,
        recommendations,
    )
    for target_product_id, recommendations in zip(
        validation_evaluation["target_product_id"],
        validation_evaluation["recommendations"],
    )
]

validation_evaluation["hit_at_10"] = (
    validation_evaluation["target_rank"].notna()
)

validation_evaluation["reciprocal_rank"] = (
    1
    / validation_evaluation["target_rank"]
).fillna(0)

validation_evaluation["ndcg_at_10"] = (
    1
    / np.log2(
        validation_evaluation["target_rank"]
        + 1
    )
).fillna(0)

known_development_products = set(
    product_popularity["parent_asin"]
)

validation_evaluation[
    "target_known_during_training"
] = validation_evaluation[
    "target_product_id"
].isin(known_development_products)

recommended_products = {
    product_id
    for recommendations in validation_evaluation[
        "recommendations"
    ]
    for product_id in recommendations
}

popularity_metrics = pd.DataFrame(
    {
        "metric": [
            "Hit Rate@10",
            "MRR@10",
            "NDCG@10",
            "Validation target catalogue availability",
            "Recommendation catalogue coverage",
        ],
        "value": [
            validation_evaluation[
                "hit_at_10"
            ].mean(),
            validation_evaluation[
                "reciprocal_rank"
            ].mean(),
            validation_evaluation[
                "ndcg_at_10"
            ].mean(),
            validation_evaluation[
                "target_known_during_training"
            ].mean(),
            (
                len(recommended_products)
                / len(known_development_products)
            ),
        ],
    }
)

assert validation_evaluation["user_id"].is_unique
assert len(validation_evaluation) == 88_124

print(
    "Validation users evaluated:",
    f"{len(validation_evaluation):,}",
)

print(
    "Unique products recommended:",
    f"{len(recommended_products):,}",
)

display(popularity_metrics)
display(validation_evaluation.head())

Validation users evaluated: 88,124
Unique products recommended: 18


,metric,value
0,Hit Rate@10,0.030650
1,MRR@10,0.011022
2,NDCG@10,0.015515
3,Validation target catalogue availability,0.996437
4,Recommendation catalogue coverage,0.000711


,user_id,target_product_id,recommendations,target_rank,hit_at_10,reciprocal_rank,ndcg_at_10,target_known_during_training
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,B07QX99XJJ,"[B01N3ASPNV, B07YBXFDYN, B0086VPUHI, B00BGA9WK...",NaN,False,0.0,0.0,True
1,AE2252DKW4XJIZP5QPFMQVJBVRTA,B07JH3LSHN,"[B01N3ASPNV, B07YBXFDYN, B0086VPUHI, B00BGA9WK...",NaN,False,0.0,0.0,True
2,AE225O22SA7DLBOGOEIFL7FT5VYQ,B00136MBHA,"[B01N3ASPNV, B07YBXFDYN, B0086VPUHI, B00BGA9WK...",NaN,False,0.0,0.0,True
3,AE227CCN4C37WTOB3J2TZPOKLEQQ,B000WCCURW,"[B01N3ASPNV, B07YBXFDYN, B0086VPUHI, B00BGA9WK...",NaN,False,0.0,0.0,True
4,AE22BPPZGGRTSYOHK2J3LCG5HGAQ,B00KVOVBGM,"[B01N3ASPNV, B07YBXFDYN, B0086VPUHI, B00BGA9WK...",NaN,False,0.0,0.0,True


### Popularity-baseline results

The popularity baseline achieved a Hit Rate@10 of 3.065%, MRR@10 of 0.0110, and NDCG@10 of 0.0155 on 88,124 validation users.

Although 99.64% of validation target products were present in development training, the baseline recommended only 18 unique products. This represents 0.0711% of the 25,319-product development catalogue.

The results demonstrate the central limitation of a global popularity recommender: it can recover some widely popular targets, but it provides almost identical recommendations to different users and exposes very little of the catalogue.

This baseline establishes the minimum ranking performance that personalized models should improve upon. It also remains useful as a fallback for users with insufficient behavioural history.

## Collaborative item-similarity model

The collaborative model learns product relationships from user behaviour.

A sparse user–product matrix is constructed from development positive interactions:

- each row represents a user;
- each column represents a product;
- a value of 1 represents an observed positive interaction;
- unobserved user–product pairs remain empty.

Products will then be compared according to the users they share. This creates personalized recommendations without requiring product descriptions or categories.

In [8]:
development_users = sorted(
    development_training["user_id"].unique()
)

development_products = sorted(
    development_training["parent_asin"].unique()
)

user_to_index = {
    user_id: index
    for index, user_id in enumerate(development_users)
}

product_to_index = {
    product_id: index
    for index, product_id in enumerate(development_products)
}

index_to_product = {
    index: product_id
    for product_id, index in product_to_index.items()
}

matrix_rows = development_training[
    "user_id"
].map(user_to_index)

matrix_columns = development_training[
    "parent_asin"
].map(product_to_index)

matrix_values = np.ones(
    len(development_training),
    dtype=np.float32,
)

user_product_matrix = csr_matrix(
    (
        matrix_values,
        (
            matrix_rows,
            matrix_columns,
        ),
    ),
    shape=(
        len(development_users),
        len(development_products),
    ),
)

assert user_product_matrix.shape == (
    len(development_users),
    len(development_products),
)

assert user_product_matrix.nnz == len(
    development_training
)

possible_matrix_positions = (
    user_product_matrix.shape[0]
    * user_product_matrix.shape[1]
)

matrix_density = (
    user_product_matrix.nnz
    / possible_matrix_positions
)

print(
    "User-product matrix shape:",
    user_product_matrix.shape,
)

print(
    "Stored positive interactions:",
    f"{user_product_matrix.nnz:,}",
)

print(
    "Possible matrix positions:",
    f"{possible_matrix_positions:,}",
)

print(
    "Matrix density:",
    f"{matrix_density:.6%}",
)

print(
    "Matrix sparsity:",
    f"{1 - matrix_density:.6%}",
)

User-product matrix shape: (93949, 25319)
Stored positive interactions: 478,619
Possible matrix positions: 2,378,694,731
Matrix density: 0.020121%
Matrix sparsity: 99.979879%


### User–product matrix interpretation

The development matrix contains 93,949 users, 25,319 products, and 478,619 observed positive interactions.

Only 0.0201% of the 2.38 billion possible user–product positions are observed, producing 99.9799% sparsity. This extreme sparsity is typical of real recommender data: each user interacts with only a tiny portion of the available catalogue.

A compressed sparse-row matrix is therefore used to store only observed interactions rather than allocating memory for billions of empty cells.

In [9]:
from scipy.sparse import diags

item_cooccurrence = (
    user_product_matrix.T
    @ user_product_matrix
).tocsr()

item_positive_counts = np.asarray(
    user_product_matrix.sum(axis=0)
).ravel()

inverse_item_norms = np.zeros_like(
    item_positive_counts,
    dtype=np.float32,
)

nonzero_items = item_positive_counts > 0

inverse_item_norms[nonzero_items] = (
    1
    / np.sqrt(
        item_positive_counts[nonzero_items]
    )
)

normalization_matrix = diags(
    inverse_item_norms
)

item_similarity = (
    normalization_matrix
    @ item_cooccurrence
    @ normalization_matrix
).tocsr()

item_similarity.setdiag(0)
item_similarity.eliminate_zeros()

number_of_item_neighbors = 50
item_neighbors = {}

for item_index in range(
    item_similarity.shape[0]
):
    row_start = item_similarity.indptr[
        item_index
    ]

    row_end = item_similarity.indptr[
        item_index + 1
    ]

    neighbor_indices = item_similarity.indices[
        row_start:row_end
    ]

    neighbor_scores = item_similarity.data[
        row_start:row_end
    ]

    if len(neighbor_scores) > number_of_item_neighbors:
        top_positions = np.argpartition(
            neighbor_scores,
            -number_of_item_neighbors,
        )[-number_of_item_neighbors:]

        neighbor_indices = neighbor_indices[
            top_positions
        ]

        neighbor_scores = neighbor_scores[
            top_positions
        ]

    descending_order = np.argsort(
        neighbor_scores
    )[::-1]

    item_id = index_to_product[
        item_index
    ]

    item_neighbors[item_id] = [
        (
            index_to_product[
                neighbor_indices[position]
            ],
            float(
                neighbor_scores[position]
            ),
        )
        for position in descending_order
    ]

products_with_neighbors = sum(
    bool(neighbors)
    for neighbors in item_neighbors.values()
)

print(
    "Nonzero product-similarity pairs:",
    f"{item_similarity.nnz:,}",
)

print(
    "Products with collaborative neighbors:",
    f"{products_with_neighbors:,}",
)

print(
    "Maximum stored neighbors per product:",
    number_of_item_neighbors,
)

sample_product_id = popularity_product_order[0]

print(
    "Sample product:",
    sample_product_id,
)

display(
    pd.DataFrame(
        item_neighbors[sample_product_id][:10],
        columns=[
            "similar_product_id",
            "collaborative_similarity",
        ],
    )
)

Nonzero product-similarity pairs: 4,740,082
Products with collaborative neighbors: 25,288
Maximum stored neighbors per product: 50
Sample product: B01N3ASPNV


,similar_product_id,collaborative_similarity
0,B087NNPYP3,0.134171
1,B081243BT6,0.114280
2,B07624RBWB,0.110196
3,B087NN2K41,0.105822
4,B087SHFL9B,0.086072
5,B08L6L6KQL,0.081654
6,B07D53QSMK,0.073115
7,B07CV6LH3V,0.068260
8,B01MY9JB2U,0.066291
9,B08D3XL1KF,0.063786


### Collaborative similarity construction

The item-similarity calculation produced 4,740,082 nonzero directional relationships. Of the 25,319 development products, 25,288 have at least one collaborative neighbour; only 31 products lack a behavioural neighbour.

For efficient recommendation, at most 50 of the strongest neighbours are retained for each product. Similarity is cosine-normalized so that globally popular products are not treated as related to every product merely because they have many interactions.

The collaborative similarity scores are behavioural associations, not probabilities. Higher values indicate stronger overlap between the users who positively interacted with the two products.

In [10]:
def recommend_collaborative_products(
    user_id,
    number_of_recommendations=10,
):
    seen_products = development_seen_by_user.get(
        user_id,
        set(),
    )

    candidate_scores = {}

    for seen_product_id in seen_products:
        neighbors = item_neighbors.get(
            seen_product_id,
            [],
        )

        for candidate_id, similarity in neighbors:
            if candidate_id in seen_products:
                continue

            candidate_scores[candidate_id] = (
                candidate_scores.get(
                    candidate_id,
                    0.0,
                )
                + similarity
            )

    ranked_candidates = sorted(
        candidate_scores.items(),
        key=lambda item: (
            -item[1],
            item[0],
        ),
    )

    recommendations = [
        product_id
        for product_id, score in ranked_candidates[
            :number_of_recommendations
        ]
    ]

    if len(recommendations) < number_of_recommendations:
        for product_id in popularity_product_order:
            if product_id in seen_products:
                continue

            if product_id in recommendations:
                continue

            recommendations.append(product_id)

            if len(recommendations) == number_of_recommendations:
                break

    return recommendations


collaborative_validation = (
    validation_targets[
        [
            "user_id",
            "parent_asin",
        ]
    ]
    .rename(
        columns={
            "parent_asin": "target_product_id"
        }
    )
    .copy()
)

collaborative_validation["recommendations"] = [
    recommend_collaborative_products(
        user_id,
        recommendation_count,
    )
    for user_id in collaborative_validation["user_id"]
]

collaborative_validation["target_rank"] = [
    find_target_rank(
        target_product_id,
        recommendations,
    )
    for target_product_id, recommendations in zip(
        collaborative_validation["target_product_id"],
        collaborative_validation["recommendations"],
    )
]

collaborative_validation["hit_at_10"] = (
    collaborative_validation["target_rank"].notna()
)

collaborative_validation["reciprocal_rank"] = (
    1
    / collaborative_validation["target_rank"]
).fillna(0)

collaborative_validation["ndcg_at_10"] = (
    1
    / np.log2(
        collaborative_validation["target_rank"]
        + 1
    )
).fillna(0)

collaborative_recommended_products = {
    product_id
    for recommendations in collaborative_validation[
        "recommendations"
    ]
    for product_id in recommendations
}

collaborative_metrics = pd.DataFrame(
    {
        "metric": [
            "Hit Rate@10",
            "MRR@10",
            "NDCG@10",
            "Recommendation catalogue coverage",
        ],
        "value": [
            collaborative_validation[
                "hit_at_10"
            ].mean(),
            collaborative_validation[
                "reciprocal_rank"
            ].mean(),
            collaborative_validation[
                "ndcg_at_10"
            ].mean(),
            (
                len(collaborative_recommended_products)
                / len(known_development_products)
            ),
        ],
    }
)

metric_comparison = pd.concat(
    [
        popularity_metrics.assign(
            model="Popularity baseline"
        ),
        collaborative_metrics.assign(
            model="Collaborative + popularity fallback"
        ),
    ],
    ignore_index=True,
)

print(
    "Validation users evaluated:",
    f"{len(collaborative_validation):,}",
)

print(
    "Unique collaborative products recommended:",
    f"{len(collaborative_recommended_products):,}",
)

display(
    metric_comparison.pivot(
        index="metric",
        columns="model",
        values="value",
    )
)

display(
    collaborative_validation.head()
)

Validation users evaluated: 88,124
Unique collaborative products recommended: 24,704


model,Collaborative + popularity fallback,Popularity baseline
metric,,
Hit Rate@10,0.052994,0.030650
MRR@10,0.023664,0.011022
NDCG@10,0.030525,0.015515
Recommendation catalogue coverage,0.975710,0.000711
Validation target catalogue availability,NaN,0.996437


,user_id,target_product_id,recommendations,target_rank,hit_at_10,reciprocal_rank,ndcg_at_10
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,B07QX99XJJ,"[B089VVH8HC, B094XXLRV9, B0B789RDJZ, B073X4RGF...",NaN,False,0.0,0.0
1,AE2252DKW4XJIZP5QPFMQVJBVRTA,B07JH3LSHN,"[B0B7KFFKT5, B00002SUVM, B000G810DC, B000006OW...",NaN,False,0.0,0.0
2,AE225O22SA7DLBOGOEIFL7FT5VYQ,B00136MBHA,"[B00400OFG6, B0009J8WZC, B0012NWBH4, B0000695H...",NaN,False,0.0,0.0
3,AE227CCN4C37WTOB3J2TZPOKLEQQ,B000WCCURW,"[B00EU5VY60, B00027IOEU, B003JKL3MW, B0049M8OY...",NaN,False,0.0,0.0
4,AE22BPPZGGRTSYOHK2J3LCG5HGAQ,B00KVOVBGM,"[B00KVOVBGM, B0178FWY62, B00WVCITRQ, B00LH6CBE...",1.0,True,1.0,1.0


### Collaborative-model results

The collaborative item-similarity recommender improved Hit Rate@10 from 3.065% to 5.299%, an approximate relative improvement of 72.9% over the popularity baseline.

MRR@10 improved by approximately 114.7%, while NDCG@10 improved by approximately 96.4%. The collaborative model therefore recovered more hidden products and generally ranked successful matches higher.

Catalogue coverage increased from 18 products (0.0711%) to 24,704 products (97.57%). Unlike the global popularity baseline, collaborative recommendations vary substantially according to each user's previous positive interactions.

These results demonstrate that behavioural personalization adds measurable value. However, collaborative filtering still cannot understand products with no development interactions and cannot explain recommendations using product characteristics. A content component is added next to address these limitations.

In [11]:
metadata_path = (
    project_root
    / "data"
    / "raw"
    / "meta_Video_Games.jsonl.gz"
)

metadata = pd.read_json(
    metadata_path,
    lines=True,
    compression="gzip",
)

interaction_catalogue = set(
    development_products
) | set(
    validation_targets["parent_asin"]
) | set(
    test_targets["parent_asin"]
)

product_catalogue = (
    metadata[
        metadata["parent_asin"].isin(
            interaction_catalogue
        )
    ]
    .drop_duplicates(
        subset="parent_asin",
        keep="first",
    )
    .copy()
)


def flatten_text(value):
    if isinstance(value, list):
        return " ".join(
            flatten_text(item)
            for item in value
        )

    if isinstance(value, dict):
        return " ".join(
            flatten_text(item)
            for item in value.values()
        )

    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value)


def extract_image_url(images):
    if not isinstance(images, list):
        return None

    for image in images:
        if not isinstance(image, dict):
            continue

        for image_size in [
            "large",
            "hi_res",
            "thumb",
        ]:
            image_url = image.get(image_size)

            if image_url:
                return image_url

    return None


text_columns = [
    "title",
    "main_category",
    "categories",
    "features",
    "description",
]

product_catalogue["content_text"] = (
    product_catalogue[text_columns]
    .apply(
        lambda row: " ".join(
            flatten_text(value)
            for value in row
        ),
        axis=1,
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

product_catalogue["image_url"] = (
    product_catalogue["images"].apply(
        extract_image_url
    )
)

product_catalogue["numeric_price"] = pd.to_numeric(
    product_catalogue["price"],
    errors="coerce",
)

catalogue_output_path = (
    processed_data_directory
    / "product_catalogue.csv.gz"
)

catalogue_columns = [
    "parent_asin",
    "title",
    "main_category",
    "categories",
    "features",
    "description",
    "numeric_price",
    "image_url",
    "content_text",
]

product_catalogue[
    catalogue_columns
].to_csv(
    catalogue_output_path,
    index=False,
    compression="gzip",
)

assert product_catalogue[
    "parent_asin"
].is_unique

assert catalogue_output_path.exists()

print(
    "Products in application catalogue:",
    f"{len(product_catalogue):,}",
)

print(
    "Products with usable content text:",
    f"{product_catalogue['content_text'].ne('').sum():,}",
)

print(
    "Products with image URLs:",
    f"{product_catalogue['image_url'].notna().sum():,}",
)

print(
    "Products with numeric prices:",
    f"{product_catalogue['numeric_price'].notna().sum():,}",
)

display(
    product_catalogue[
        [
            "parent_asin",
            "title",
            "main_category",
            "numeric_price",
            "image_url",
        ]
    ].head()
)

Products in application catalogue: 25,585
Products with usable content text: 25,585
Products with image URLs: 25,584
Products with numeric prices: 17,114


,parent_asin,title,main_category,numeric_price,image_url
2,B00Z9TLVK0,NBA 2K17 - Early Tip Off Edition - PlayStation 4,Video Games,58.00,https://m.media-amazon.com/images/I/51wlIPcf0g...
7,B07H93H878,eXtremeRate Soft Touch Top Shell Front Housing...,Video Games,17.59,https://m.media-amazon.com/images/I/41reviTraV...
9,B00006969T,Konami Collector's Series: Castlevania & Contr...,Video Games,NaN,https://m.media-amazon.com/images/I/21CERVWQ5D...
10,B07DL353F8,Tales of Vesperia - Definitive Edition - PlayS...,Video Games,20.95,https://m.media-amazon.com/images/I/51vhZgBs+0...
11,B00BJH85SW,Turbo: Super Stunt Squad - Nintendo 3DS,Video Games,4.97,https://m.media-amazon.com/images/I/61UmkQluCp...


### Product metadata coverage

The application catalogue contains 25,585 products appearing in the positive development, validation, or test data.

All catalogue products have usable text for content modelling, and all but one have an image URL. Exact numeric prices are available for 17,114 products; missing or non-exact prices will be presented as unavailable rather than interpreted as zero.

The metadata allows the system to represent products that have limited behavioural history and enables the Streamlit application to display human-readable titles, images, categories, and prices instead of internal product identifiers.

In [14]:
content_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=30_000,
    sublinear_tf=True,
)

content_matrix = content_vectorizer.fit_transform(
    product_catalogue["content_text"]
)

content_neighbor_model = NearestNeighbors(
    n_neighbors=51,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

content_neighbor_model.fit(
    content_matrix
)

content_distances, content_indices = (
    content_neighbor_model.kneighbors(
        content_matrix
    )
)

catalogue_product_ids = (
    product_catalogue["parent_asin"]
    .reset_index(drop=True)
    .to_numpy()
)

content_neighbors = {}

for product_position, product_id in enumerate(
    catalogue_product_ids
):
    neighbors = []

    for neighbor_position, distance in zip(
        content_indices[product_position],
        content_distances[product_position],
    ):
        neighbor_id = catalogue_product_ids[
            neighbor_position
        ]

        if neighbor_id == product_id:
            continue

        similarity = 1 - float(distance)

        neighbors.append(
            (
                neighbor_id,
                similarity,
            )
        )

        if len(neighbors) == 50:
            break

    content_neighbors[product_id] = neighbors

assert content_matrix.shape[0] == len(
    product_catalogue
)

print(
    "Content matrix shape:",
    content_matrix.shape,
)

print(
    "TF-IDF vocabulary size:",
    f"{len(content_vectorizer.vocabulary_):,}",
)

print(
    "Products with content neighbors:",
    f"{sum(bool(value) for value in content_neighbors.values()):,}",
)

sample_content_product = (
    product_catalogue.iloc[0]["parent_asin"]
)

sample_neighbor_ids = [
    product_id
    for product_id, similarity in content_neighbors[
        sample_content_product
    ][:10]
]

sample_content_results = (
    product_catalogue[
        product_catalogue["parent_asin"].isin(
            sample_neighbor_ids
        )
    ][
        [
            "parent_asin",
            "title",
            "main_category",
        ]
    ]
    .copy()
)

sample_similarity_lookup = dict(
    content_neighbors[sample_content_product]
)

sample_content_results[
    "content_similarity"
] = sample_content_results[
    "parent_asin"
].map(sample_similarity_lookup)

sample_content_results = (
    sample_content_results
    .sort_values(
        "content_similarity",
        ascending=False,
    )
)

sample_title = product_catalogue.loc[
    product_catalogue["parent_asin"]
    == sample_content_product,
    "title",
].iloc[0]

print(
    "Sample source product:",
    sample_title,
)

display(
    sample_content_results
)

Content matrix shape: (25585, 30000)
TF-IDF vocabulary size: 30,000
Products with content neighbors: 25,585
Sample source product: NBA 2K17 - Early Tip Off Edition - PlayStation 4


,parent_asin,title,main_category,content_similarity
469,B01MG6DORB,NBA 2K17 Standard Edition - PlayStation 4,Video Games,0.627535
15359,B07BT5L9RS,NBA 2K17 - Legends Gold - Xbox One Digital Code,Video Games,0.483824
18065,B09B4RT164,NBA 2K22 75th Anniversary Edition - PlayStation 4,Video Games,0.354483
102525,B071YZVS75,NBA 2K18 Standard Edition - Nintendo Switch,Video Games,0.342309
158,B0B6BF97TZ,NBA 2K23 Michael Jordan Edition - Xbox Series X,Video Games,0.313937
8454,B087SLTR2B,NBA 2K20 Legend Edition Xbox One,Video Games,0.304505
9575,B08CG1GDR2,NBA 2K21 Mamba Forever Edition - Nintendo Switch,Video Games,0.291439
4365,B076541PNM,Nba 2K18 Standard Edition - Xbox One,Video Games,0.286744
11727,B077GCYVD7,NBA 2K18 Early Tip-Off Edition - PlayStation 4,Video Games,0.286671
71096,B071L43HFZ,NBA 2K18 - PS4 [Digital Code],Video Games,0.286232


### Content-similarity results

The TF-IDF representation contains 25,585 products and 30,000 textual features. Every catalogue product received content-based neighbours.

A qualitative inspection using *NBA 2K17 – Early Tip Off Edition – PlayStation 4* returned other NBA games, editions, years, and platforms. This indicates that the content model captures meaningful product semantics rather than global popularity alone.

Content similarity also supports item cold start because a product can receive neighbours from its metadata even when it has limited interaction history. Its limitation is that textual similarity does not necessarily represent user preference, so it is combined with behavioural collaborative evidence.

In [19]:
def recommend_hybrid_products(
    user_id,
    number_of_recommendations=10,
    collaborative_weight=0.70,
):
    content_weight = 1 - collaborative_weight

    seen_products = development_seen_by_user.get(
        user_id,
        set(),
    )

    collaborative_scores = {}
    content_scores = {}

    for seen_product_id in seen_products:
        for candidate_id, similarity in item_neighbors.get(
            seen_product_id,
            [],
        ):
            if candidate_id not in seen_products:
                collaborative_scores[candidate_id] = (
                    collaborative_scores.get(
                        candidate_id,
                        0.0,
                    )
                    + similarity
                )

        for candidate_id, similarity in content_neighbors.get(
            seen_product_id,
            [],
        ):
            if candidate_id not in seen_products:
                content_scores[candidate_id] = (
                    content_scores.get(
                        candidate_id,
                        0.0,
                    )
                    + similarity
                )

    maximum_collaborative_score = max(
    collaborative_scores.values(),
    default=0.0,
    )

    maximum_content_score = max(
        content_scores.values(),
        default=0.0,
    )

    if maximum_collaborative_score <= 0:
        maximum_collaborative_score = 1.0

    if maximum_content_score <= 0:
        maximum_content_score = 1.0

    candidate_ids = (
        set(collaborative_scores)
        | set(content_scores)
    )

    hybrid_scores = {}

    for candidate_id in candidate_ids:
        normalized_collaborative_score = (
            collaborative_scores.get(
                candidate_id,
                0.0,
            )
            / maximum_collaborative_score
        )

        normalized_content_score = (
            content_scores.get(
                candidate_id,
                0.0,
            )
            / maximum_content_score
        )

        hybrid_scores[candidate_id] = (
            collaborative_weight
            * normalized_collaborative_score
            + content_weight
            * normalized_content_score
        )

    ranked_candidates = sorted(
        hybrid_scores.items(),
        key=lambda item: (
            -item[1],
            item[0],
        ),
    )

    recommendations = [
        product_id
        for product_id, score in ranked_candidates[
            :number_of_recommendations
        ]
    ]

    if len(recommendations) < number_of_recommendations:
        for product_id in popularity_product_order:
            if product_id in seen_products:
                continue

            if product_id in recommendations:
                continue

            recommendations.append(product_id)

            if len(recommendations) == number_of_recommendations:
                break

    return recommendations


sample_user_id = validation_targets.iloc[0][
    "user_id"
]

sample_user_history_ids = list(
    development_seen_by_user[sample_user_id]
)

sample_hybrid_ids = recommend_hybrid_products(
    sample_user_id,
    number_of_recommendations=10,
    collaborative_weight=0.70,
)

sample_history = (
    product_catalogue[
        product_catalogue["parent_asin"].isin(
            sample_user_history_ids
        )
    ][
        [
            "parent_asin",
            "title",
        ]
    ]
)

sample_hybrid_recommendations = (
    product_catalogue[
        product_catalogue["parent_asin"].isin(
            sample_hybrid_ids
        )
    ][
        [
            "parent_asin",
            "title",
            "main_category",
            "numeric_price",
        ]
    ]
    .copy()
)

recommendation_order = {
    product_id: rank
    for rank, product_id in enumerate(
        sample_hybrid_ids,
        start=1,
    )
}

sample_hybrid_recommendations["rank"] = (
    sample_hybrid_recommendations[
        "parent_asin"
    ].map(recommendation_order)
)

sample_hybrid_recommendations = (
    sample_hybrid_recommendations
    .sort_values("rank")
)

print("Sample user:", sample_user_id)
print()
print("Products in this user's development history:")

display(sample_history)

print("Hybrid recommendations:")

display(sample_hybrid_recommendations)

Sample user: AE222HFZDH6BPTYFOUWGGU63YSIQ

Products in this user's development history:


,parent_asin,title
6419,B01GY35T4S,Xbox One S 1TB Console - NBA 2K20 Bundle - [DI...
64932,B07SNN8GV5,Call of Duty: Modern Warfare Battle Pass Editi...
66612,B082R1RGZF,Turtle Beach Stealth 300 Amplified Surround So...


Hybrid recommendations:


,parent_asin,title,main_category,numeric_price,rank
26350,B089VVH8HC,"YARENKA Replacement Astro A10 A40 Cable, 2.0M ...",Industrial & Scientific,19.99,1
2649,B094XXLRV9,Playrealm FPS Thumbstick Extender & Printing R...,Video Games,12.99,2
43684,B0B789RDJZ,HyperX ChargePlay Duo - Controller Charging St...,Computers,NaN,3
64948,B073X4RGFP,"PDP Gaming 3-In-1 Folio Screen Protector, Stan...",Video Games,NaN,4
76852,B07CLZMCYK,Talentech Ember Plus Ergonomic RGB USB Wired G...,Cell Phones & Accessories,NaN,5
34825,B06WLMX6N5,PowerA Fusion Wired Stereo Gaming Headset with...,Video Games,43.54,6
30439,B07X97WQS4,Xbox All Access - Xbox One X 1TB Console - Gea...,Video Games,NaN,7
74566,B07GN8YD2H,"Gaming Headset for PC, PS4, Xbox One, Stereo G...",Computers,NaN,8
37105,B07H2BHLGX,iMW Wired Gaming Controller for NES Classic Ed...,Video Games,14.07,9
71881,B0B3GHCTQB,Call of Duty: Modern Warfare II - PlayStation 4,Video Games,NaN,10


### Qualitative hybrid inspection

For the sampled user, the development history includes an Xbox NBA console bundle, a Call of Duty product, and a Turtle Beach gaming headset.

The hybrid recommendations contain another Call of Duty title, an Xbox console, controllers, charging accessories, and gaming headsets. This demonstrates that the model combines behavioural associations with product-description similarity.

The inspection also exposes metadata noise: some gaming accessories are assigned broad categories such as `Industrial & Scientific`, `Computers`, or `Cell Phones & Accessories`. These results should not be silently removed from the evidence. They show that source metadata quality can influence recommendation quality and motivate category-aware filtering or reranking as a future improvement.

In [20]:

hybrid_weights_to_test = [
    0.30,
    0.50,
    0.70,
    0.90,
]

validation_user_targets = list(
    validation_targets[
        [
            "user_id",
            "parent_asin",
        ]
    ].itertuples(
        index=False,
        name=None,
    )
)

hybrid_weight_results = []

for collaborative_weight in hybrid_weights_to_test:
    start_time = time.time()

    target_ranks = []
    unique_recommended_products = set()

    for user_id, target_product_id in validation_user_targets:
        recommendations = recommend_hybrid_products(
            user_id,
            number_of_recommendations=10,
            collaborative_weight=collaborative_weight,
        )

        unique_recommended_products.update(
            recommendations
        )

        target_rank = find_target_rank(
            target_product_id,
            recommendations,
        )

        target_ranks.append(target_rank)

    target_ranks = np.asarray(
        target_ranks,
        dtype=float,
    )

    successful_recommendations = ~np.isnan(
        target_ranks
    )

    reciprocal_ranks = np.zeros(
        len(target_ranks),
        dtype=float,
    )

    reciprocal_ranks[
        successful_recommendations
    ] = (
        1
        / target_ranks[
            successful_recommendations
        ]
    )

    ndcg_scores = np.zeros(
        len(target_ranks),
        dtype=float,
    )

    ndcg_scores[
        successful_recommendations
    ] = (
        1
        / np.log2(
            target_ranks[
                successful_recommendations
            ]
            + 1
        )
    )

    hybrid_weight_results.append(
        {
            "collaborative_weight": collaborative_weight,
            "content_weight": 1 - collaborative_weight,
            "hit_rate_at_10": successful_recommendations.mean(),
            "mrr_at_10": reciprocal_ranks.mean(),
            "ndcg_at_10": ndcg_scores.mean(),
            "catalogue_coverage": (
                len(unique_recommended_products)
                / len(product_catalogue)
            ),
            "unique_products_recommended": len(
                unique_recommended_products
            ),
            "runtime_seconds": (
                time.time()
                - start_time
            ),
        }
    )

    print(
        "Finished collaborative weight:",
        collaborative_weight,
    )

hybrid_weight_results = pd.DataFrame(
    hybrid_weight_results
).sort_values(
    [
        "ndcg_at_10",
        "hit_rate_at_10",
    ],
    ascending=False,
)

selected_collaborative_weight = (
    hybrid_weight_results.iloc[0][
        "collaborative_weight"
    ]
)

selected_content_weight = (
    1 - selected_collaborative_weight
)

print()
print(
    "Selected collaborative weight:",
    selected_collaborative_weight,
)

print(
    "Selected content weight:",
    selected_content_weight,
)

display(
    hybrid_weight_results
)

Finished collaborative weight: 0.3
Finished collaborative weight: 0.5
Finished collaborative weight: 0.7
Finished collaborative weight: 0.9

Selected collaborative weight: 0.7
Selected content weight: 0.30000000000000004


,collaborative_weight,content_weight,hit_rate_at_10,mrr_at_10,ndcg_at_10,catalogue_coverage,unique_products_recommended,runtime_seconds
2,0.7,0.3,0.059530,0.026575,0.034275,0.972562,24883,17.092799
1,0.5,0.5,0.055603,0.024626,0.031850,0.974868,24942,16.902082
3,0.9,0.1,0.055150,0.024547,0.031701,0.967676,24758,17.791726
0,0.3,0.7,0.041271,0.016046,0.021895,0.936252,23954,16.390844


### Hybrid weight selection

Four collaborative/content weight combinations were compared using validation targets only.

The 70% collaborative and 30% content configuration achieved the highest validation NDCG@10, Hit Rate@10, and MRR@10. It improved Hit Rate@10 by approximately 12.3% relative to the collaborative model with popularity fallback, while recommending 24,883 unique products.

Content-heavy weighting performed worse, indicating that behavioural co-preference is more predictive than textual similarity for this dataset. Nevertheless, a 30% content contribution improved the collaborative ranking and supports products with limited behavioural evidence.

The 70/30 configuration is therefore frozen before test evaluation. No weight will be changed in response to the test results.

In [21]:
def build_collaborative_artifacts(
    training_table,
    maximum_neighbors=50,
):
    users = sorted(
        training_table["user_id"].unique()
    )

    products = sorted(
        training_table["parent_asin"].unique()
    )

    user_index = {
        user_id: index
        for index, user_id in enumerate(users)
    }

    product_index = {
        product_id: index
        for index, product_id in enumerate(products)
    }

    product_from_index = {
        index: product_id
        for product_id, index in product_index.items()
    }

    row_indices = training_table[
        "user_id"
    ].map(user_index)

    column_indices = training_table[
        "parent_asin"
    ].map(product_index)

    values = np.ones(
        len(training_table),
        dtype=np.float32,
    )

    interaction_matrix = csr_matrix(
        (
            values,
            (
                row_indices,
                column_indices,
            ),
        ),
        shape=(
            len(users),
            len(products),
        ),
    )

    cooccurrence = (
        interaction_matrix.T
        @ interaction_matrix
    ).tocsr()

    product_counts = np.asarray(
        interaction_matrix.sum(axis=0)
    ).ravel()

    inverse_norms = np.zeros_like(
        product_counts,
        dtype=np.float32,
    )

    positive_count_mask = product_counts > 0

    inverse_norms[positive_count_mask] = (
        1
        / np.sqrt(
            product_counts[
                positive_count_mask
            ]
        )
    )

    normalization = diags(
        inverse_norms
    )

    similarity_matrix = (
        normalization
        @ cooccurrence
        @ normalization
    ).tocsr()

    similarity_matrix.setdiag(0)
    similarity_matrix.eliminate_zeros()

    neighbor_lookup = {}

    for product_position in range(
        similarity_matrix.shape[0]
    ):
        start = similarity_matrix.indptr[
            product_position
        ]

        end = similarity_matrix.indptr[
            product_position + 1
        ]

        neighbor_positions = (
            similarity_matrix.indices[
                start:end
            ]
        )

        neighbor_scores = (
            similarity_matrix.data[
                start:end
            ]
        )

        if len(neighbor_scores) > maximum_neighbors:
            strongest_positions = np.argpartition(
                neighbor_scores,
                -maximum_neighbors,
            )[-maximum_neighbors:]

            neighbor_positions = neighbor_positions[
                strongest_positions
            ]

            neighbor_scores = neighbor_scores[
                strongest_positions
            ]

        score_order = np.argsort(
            neighbor_scores
        )[::-1]

        product_id = product_from_index[
            product_position
        ]

        neighbor_lookup[product_id] = [
            (
                product_from_index[
                    neighbor_positions[position]
                ],
                float(
                    neighbor_scores[position]
                ),
            )
            for position in score_order
        ]

    seen_lookup = (
        training_table
        .groupby("user_id")["parent_asin"]
        .agg(set)
        .to_dict()
    )

    popularity_order = (
        training_table
        .groupby("parent_asin")
        .size()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    return {
        "users": users,
        "products": products,
        "seen_by_user": seen_lookup,
        "popularity_order": popularity_order,
        "item_neighbors": neighbor_lookup,
        "matrix_shape": interaction_matrix.shape,
        "stored_interactions": interaction_matrix.nnz,
        "similarity_pairs": similarity_matrix.nnz,
    }


final_artifacts = build_collaborative_artifacts(
    final_training,
    maximum_neighbors=50,
)

print(
    "Final matrix shape:",
    final_artifacts["matrix_shape"],
)

print(
    "Final stored interactions:",
    f"{final_artifacts['stored_interactions']:,}",
)

print(
    "Final collaborative similarity pairs:",
    f"{final_artifacts['similarity_pairs']:,}",
)

print(
    "Final users:",
    f"{len(final_artifacts['users']):,}",
)

print(
    "Final products:",
    f"{len(final_artifacts['products']):,}",
)

Final matrix shape: (93949, 25499)
Final stored interactions: 566,743
Final collaborative similarity pairs: 5,379,976
Final users: 93,949
Final products: 25,499


## Final test evaluation

After selecting the 70% collaborative and 30% content configuration using validation data, the collaborative component is rebuilt using final training data.

Final training includes the former validation interactions but continues to exclude all test targets. The selected configuration is now evaluated once against the untouched test set.

The test result estimates how the frozen recommender generalizes to later unseen positive interactions. No model settings will be changed in response to this result.

In [22]:
def recommend_from_artifacts(
    user_id,
    artifacts,
    number_of_recommendations=10,
    collaborative_weight=0.70,
):
    content_weight = 1 - collaborative_weight

    seen_products = artifacts[
        "seen_by_user"
    ].get(
        user_id,
        set(),
    )

    collaborative_scores = {}
    content_scores = {}

    for seen_product_id in seen_products:
        for candidate_id, similarity in artifacts[
            "item_neighbors"
        ].get(
            seen_product_id,
            [],
        ):
            if candidate_id not in seen_products:
                collaborative_scores[candidate_id] = (
                    collaborative_scores.get(
                        candidate_id,
                        0.0,
                    )
                    + similarity
                )

        for candidate_id, similarity in content_neighbors.get(
            seen_product_id,
            [],
        ):
            if candidate_id not in seen_products:
                content_scores[candidate_id] = (
                    content_scores.get(
                        candidate_id,
                        0.0,
                    )
                    + similarity
                )

    maximum_collaborative_score = max(
        collaborative_scores.values(),
        default=0.0,
    )

    maximum_content_score = max(
        content_scores.values(),
        default=0.0,
    )

    if maximum_collaborative_score <= 0:
        maximum_collaborative_score = 1.0

    if maximum_content_score <= 0:
        maximum_content_score = 1.0

    candidate_ids = (
        set(collaborative_scores)
        | set(content_scores)
    )

    hybrid_scores = {}

    for candidate_id in candidate_ids:
        normalized_collaborative_score = (
            collaborative_scores.get(
                candidate_id,
                0.0,
            )
            / maximum_collaborative_score
        )

        normalized_content_score = (
            content_scores.get(
                candidate_id,
                0.0,
            )
            / maximum_content_score
        )

        hybrid_scores[candidate_id] = (
            collaborative_weight
            * normalized_collaborative_score
            + content_weight
            * normalized_content_score
        )

    ranked_candidates = sorted(
        hybrid_scores.items(),
        key=lambda item: (
            -item[1],
            item[0],
        ),
    )

    recommendations = [
        product_id
        for product_id, score in ranked_candidates[
            :number_of_recommendations
        ]
    ]

    if len(recommendations) < number_of_recommendations:
        for product_id in artifacts[
            "popularity_order"
        ]:
            if product_id in seen_products:
                continue

            if product_id in recommendations:
                continue

            recommendations.append(product_id)

            if len(recommendations) == number_of_recommendations:
                break

    return recommendations


test_user_targets = list(
    test_targets[
        [
            "user_id",
            "parent_asin",
        ]
    ].itertuples(
        index=False,
        name=None,
    )
)

test_ranks = []
test_recommendation_lists = []
test_recommended_products = set()

test_start_time = time.time()

for user_id, target_product_id in test_user_targets:
    recommendations = recommend_from_artifacts(
        user_id,
        final_artifacts,
        number_of_recommendations=10,
        collaborative_weight=selected_collaborative_weight,
    )

    test_recommendation_lists.append(
        recommendations
    )

    test_recommended_products.update(
        recommendations
    )

    test_ranks.append(
        find_target_rank(
            target_product_id,
            recommendations,
        )
    )

test_results = test_targets[
    [
        "user_id",
        "parent_asin",
    ]
].rename(
    columns={
        "parent_asin": "target_product_id"
    }
).copy()

test_results["recommendations"] = (
    test_recommendation_lists
)

test_results["target_rank"] = np.asarray(
    test_ranks,
    dtype=float,
)

test_results["hit_at_10"] = (
    test_results["target_rank"].notna()
)

test_results["reciprocal_rank"] = 0.0
test_results.loc[
    test_results["hit_at_10"],
    "reciprocal_rank",
] = (
    1
    / test_results.loc[
        test_results["hit_at_10"],
        "target_rank",
    ]
)

test_results["ndcg_at_10"] = 0.0
test_results.loc[
    test_results["hit_at_10"],
    "ndcg_at_10",
] = (
    1
    / np.log2(
        test_results.loc[
            test_results["hit_at_10"],
            "target_rank",
        ]
        + 1
    )
)

final_known_products = set(
    final_artifacts["products"]
)

test_results[
    "target_known_collaboratively"
] = test_results[
    "target_product_id"
].isin(final_known_products)

test_metrics = pd.DataFrame(
    {
        "metric": [
            "Hit Rate@10",
            "MRR@10",
            "NDCG@10",
            "Collaborative target availability",
            "Recommendation catalogue coverage",
        ],
        "value": [
            test_results[
                "hit_at_10"
            ].mean(),
            test_results[
                "reciprocal_rank"
            ].mean(),
            test_results[
                "ndcg_at_10"
            ].mean(),
            test_results[
                "target_known_collaboratively"
            ].mean(),
            (
                len(test_recommended_products)
                / len(product_catalogue)
            ),
        ],
    }
)

assert len(test_results) == 88_124
assert test_results["user_id"].is_unique

print(
    "Test users evaluated:",
    f"{len(test_results):,}",
)

print(
    "Unique products recommended:",
    f"{len(test_recommended_products):,}",
)

print(
    "Test runtime:",
    f"{time.time() - test_start_time:.2f} seconds",
)

display(test_metrics)
display(test_results.head())

Test users evaluated: 88,124
Unique products recommended: 24,829
Test runtime: 26.15 seconds


,metric,value
0,Hit Rate@10,0.056386
1,MRR@10,0.024993
2,NDCG@10,0.032330
3,Collaborative target availability,0.997379
4,Recommendation catalogue coverage,0.970451


,user_id,target_product_id,recommendations,target_rank,hit_at_10,reciprocal_rank,ndcg_at_10,target_known_collaboratively
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,B0BW17W9GM,"[B06WLMX6N5, B073X4RGFP, B0B789RDJZ, B07CLZMCY...",NaN,False,0.0,0.0,True
1,AE2252DKW4XJIZP5QPFMQVJBVRTA,B09DH4J73J,"[B0B7KFFKT5, B09XFH6NY4, B00002SUVM, B000006OW...",NaN,False,0.0,0.0,True
2,AE225O22SA7DLBOGOEIFL7FT5VYQ,B0053BCML6,"[B00400OFG6, B000JGJ6K0, B0009J8WZC, B0012NWBH...",NaN,False,0.0,0.0,True
3,AE227CCN4C37WTOB3J2TZPOKLEQQ,B000IGE78M,"[B000B659FE, B000Q8UPKQ, B00FJS28QM, B00027IOE...",NaN,False,0.0,0.0,True
4,AE22BPPZGGRTSYOHK2J3LCG5HGAQ,B0053BCML6,"[B00MI4Y3W2, B0178FWY62, B00WVCITRQ, B091LBJKR...",NaN,False,0.0,0.0,True


### Final test results

The frozen 70% collaborative and 30% content hybrid recommender was evaluated once against 88,124 untouched test targets.

The model achieved:

- Hit Rate@10: 5.6386%;
- MRR@10: 0.024993;
- NDCG@10: 0.032330;
- recommendation catalogue coverage: 97.0451%;
- 24,829 unique recommended products.

Approximately 99.74% of test targets were present in final collaborative training. The content component provides an additional route for products with limited behavioural evidence.

Test ranking metrics were approximately 5–6% lower than validation metrics. This modest decline is expected on later untouched interactions and indicates reasonably stable generalization rather than a severe validation–test collapse.

The model should not be described as production-ready. Evaluation is offline, positive-only, and based on one historical dataset. The metadata is a current snapshot rather than a temporally versioned catalogue, and recommendation quality may be affected by noisy product categories.

In [24]:
models_directory = (
    project_root
    / "models"
)

reports_directory = (
    project_root
    / "reports"
)

models_directory.mkdir(
    parents=True,
    exist_ok=True,
)

reports_directory.mkdir(
    parents=True,
    exist_ok=True,
)

application_artifacts = {
    "seen_by_user": final_artifacts[
        "seen_by_user"
    ],
    "popularity_order": final_artifacts[
        "popularity_order"
    ],
    "item_neighbors": final_artifacts[
        "item_neighbors"
    ],
    "content_neighbors": content_neighbors,
    "collaborative_weight": float(
        selected_collaborative_weight
    ),
    "content_weight": float(
        selected_content_weight
    ),
    "recommendation_count": 10,
}

artifact_path = (
    models_directory
    / "hybrid_recommender.joblib"
)

joblib.dump(
    application_artifacts,
    artifact_path,
    compress=3,
)

validation_report = (
    hybrid_weight_results.copy()
)

validation_report.to_csv(
    reports_directory
    / "hybrid_validation_results.csv",
    index=False,
)

test_metrics.to_csv(
    reports_directory
    / "final_test_metrics.csv",
    index=False,
)

model_comparison = pd.DataFrame(
    [
        {
            "model": "Popularity baseline",
            "evaluation_split": "validation",
            "hit_rate_at_10": 0.030650,
            "mrr_at_10": 0.011022,
            "ndcg_at_10": 0.015515,
            "catalogue_coverage": 0.000711,
        },
        {
            "model": "Collaborative + popularity fallback",
            "evaluation_split": "validation",
            "hit_rate_at_10": 0.052994,
            "mrr_at_10": 0.023664,
            "ndcg_at_10": 0.030525,
            "catalogue_coverage": 0.975710,
        },
        {
            "model": "Hybrid 70/30",
            "evaluation_split": "validation",
            "hit_rate_at_10": 0.059530,
            "mrr_at_10": 0.026575,
            "ndcg_at_10": 0.034275,
            "catalogue_coverage": 0.972562,
        },
        {
            "model": "Hybrid 70/30",
            "evaluation_split": "test",
            "hit_rate_at_10": 0.056386,
            "mrr_at_10": 0.024993,
            "ndcg_at_10": 0.032330,
            "catalogue_coverage": 0.970451,
        },
    ]
)

model_comparison.to_csv(
    reports_directory
    / "model_comparison.csv",
    index=False,
)

artifact_size_mb = (
    artifact_path.stat().st_size
    / 1_000_000
)

assert artifact_path.exists()

print(
    "Saved model artifact:",
    artifact_path,
)

print(
    "Artifact size:",
    f"{artifact_size_mb:.2f} MB",
)

print()
print("Saved evaluation reports:")

for report_name in [
    "hybrid_validation_results.csv",
    "final_test_metrics.csv",
    "model_comparison.csv",
]:
    print(
        reports_directory
        / report_name
    )

Saved model artifact: /Users/trevaogwang/Documents/mobile-projects/saidia_recommeder/models/hybrid_recommender.joblib
Artifact size: 24.36 MB

Saved evaluation reports:
/Users/trevaogwang/Documents/mobile-projects/saidia_recommeder/reports/hybrid_validation_results.csv
/Users/trevaogwang/Documents/mobile-projects/saidia_recommeder/reports/final_test_metrics.csv
/Users/trevaogwang/Documents/mobile-projects/saidia_recommeder/reports/model_comparison.csv


In [25]:
application_catalogue_path = (
    models_directory
    / "product_catalogue.csv.gz"
)

product_catalogue[
    catalogue_columns
].to_csv(
    application_catalogue_path,
    index=False,
    compression="gzip",
)

print(
    "Saved application catalogue:",
    application_catalogue_path,
)

print(
    "Catalogue size:",
    f"{application_catalogue_path.stat().st_size / 1_000_000:.2f} MB",
)

Saved application catalogue: /Users/trevaogwang/Documents/mobile-projects/saidia_recommeder/models/product_catalogue.csv.gz
Catalogue size: 18.30 MB
